# ROBOT-XD++: run the revision experiments on Colab

Runs every experiment requested in the second-round review (see `docs/REVIEW_TRACEABILITY.md`), saves results to Google Drive, and draws all paper figures from the saved predictions.

* Use a GPU runtime (Runtime → Change runtime type → GPU; A100/L4 recommended).
* Every step **resumes**: if the session disconnects, re-run the cells and finished runs are skipped.
* Approximate GPU time is listed per block; run the blocks you need.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = 'https://github.com/arpitarupal2012-collab/ROBOT-Plus.git'
!git clone -q $REPO /content/ROBOT-Plus 2>/dev/null || (cd /content/ROBOT-Plus && git pull -q)
%cd /content/ROBOT-Plus
!pip -q install -r requirements.txt pydicom kaggle

DATA = '/content/drive/MyDrive/robotxd_data'      # datasets (large: keep on Drive or a mounted disk)
OUT  = '/content/drive/MyDrive/robotxd_results'   # all results, tables and figures
import os; os.makedirs(DATA, exist_ok=True); os.makedirs(OUT, exist_ok=True)

## 1. Data
Upload `kaggle.json` (Kaggle → Settings → API → Create New Token). For RSNA, first accept the competition rules on Kaggle. MIMIC-CXR needs PhysioNet credentials and is best downloaded on a machine with ~600 GB of disk; skip it here if needed.

In [ ]:
from google.colab import files
if not os.path.exists('/root/.kaggle/kaggle.json'):
    up = files.upload()  # select kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    open('/root/.kaggle/kaggle.json', 'wb').write(up['kaggle.json']); os.chmod('/root/.kaggle/kaggle.json', 0o600)

os.environ['DATA'] = DATA
!bash scripts/download_data.sh nih chexpert rsna covidx covid19rad
!python scripts/prepare_rsna.py --root $DATA/rsna-pneumonia-detection-challenge --out-annotations $OUT/annotations/rsna_boxes.csv

In [ ]:
# Dataset roots (edit if your folders differ; see docs/DATASETS.md)
NIH      = f'{DATA}/NIH_ChestXray14'
CHEXPERT = f'{DATA}/CheXpert/CheXpert-v1.0-small'
MIMIC    = f'{DATA}/physionet.org/files/mimic-cxr-jpg/2.1.0'   # optional
RSNA     = f'{DATA}/rsna-pneumonia-detection-challenge'
COVIDX   = f'{DATA}/COVIDx'
COVIDRAD = f'{DATA}/COVID-19_Radiography_Dataset'
SPLITS   = f'{OUT}/splits'
!ls $NIH $CHEXPERT $COVIDX | head

## 2. Leakage checks (R2.1, R4.2): NIH vs RSNA, COVIDx and COVID-19 Radiography

In [ ]:
# fixes the NIH split first, so the overlap is checked against the actual NIH training patients
!python -c "import sys; sys.path.insert(0,'.'); from robotxd.builders import build_splits; build_splits({'dataset':'nih','root':'$NIH','split_dir':'$SPLITS/nih_seed42'})"
for name, root in [('rsna', f'{RSNA}/png'), ('covidx', COVIDX), ('covid19rad', COVIDRAD)]:
    !python scripts/find_cross_dataset_duplicates.py --source-root $NIH --target-root $root \
        --nih-train-split $SPLITS/nih_seed42/train.csv --out $OUT/exclusions/{name}_vs_nih.csv

## 3. In-domain, patient-disjoint (R2.1, R4.1, R4.4)  ·  ~250 GPU-h

In [ ]:
S = '42 7 13 21 99'
!python scripts/run_grid.py --grid configs/grid_baselines_nih.yaml --out $OUT --set data.root=$NIH data.split_dir=$SPLITS/nih_seed42
for s in S.split():
    !python scripts/train.py --config configs/chexpert_robotxdpp.yaml --seed {s} --out $OUT/chexpert --set data.root=$CHEXPERT data.split_dir=$SPLITS/chexpert_seed42
    !python scripts/train.py --config configs/covidx_robotxdpp.yaml   --seed {s} --out $OUT/covidx --set data.root=$COVIDX data.split_dir=$SPLITS/covidx_official
!python scripts/summarize.py --root $OUT/grid_baselines_nih --reference robotxdpp --bootstrap 2000
!python scripts/summarize.py --root $OUT/chexpert --bootstrap 2000
!python scripts/summarize.py --root $OUT/covidx --bootstrap 2000

## 4. In-domain COVID-19 Radiography ablation (headline numbers, Tables 6, 7, 12)  ·  ~70 GPU-h

In [ ]:
!python scripts/run_grid.py --grid configs/grid_ablation_covid.yaml --out $OUT --set data.root=$COVIDRAD data.split_dir=$SPLITS/covid_seed42
!python scripts/summarize.py --root $OUT/grid_ablation_covid --reference full --bootstrap 2000

## 5. Priority experiment 1: shared-label transfer, source-only vs UDA  ·  ~400 GPU-h (all directions)

In [ ]:
pairs = {
  'transfer_nih_to_chexpert': f'data.root={NIH} data.split_dir={SPLITS}/nih_shared_seed42 target.root={CHEXPERT} target.split_dir={SPLITS}/chexpert_shared_seed42',
  'transfer_chexpert_to_nih': f'data.root={CHEXPERT} data.split_dir={SPLITS}/chexpert_shared_seed42 target.root={NIH} target.split_dir={SPLITS}/nih_shared_seed42',
  # 'transfer_chexpert_to_mimic': f'data.root={CHEXPERT} data.split_dir={SPLITS}/chexpert_seed42 target.root={MIMIC} target.split_dir={SPLITS}/mimic_official',
  # 'transfer_nih_to_mimic':      f'data.root={NIH} data.split_dir={SPLITS}/nih_shared_seed42 target.root={MIMIC} target.split_dir={SPLITS}/mimic_shared_official',
}
for cfg, sets in pairs.items():
    for s in S.split():
        !python scripts/train.py --config configs/{cfg}.yaml --seed {s} --out $OUT/priority1 --name {cfg}_source_only --set {sets} loss.alignment=none
        !python scripts/train.py --config configs/{cfg}.yaml --seed {s} --out $OUT/priority1 --name {cfg}_uda --set {sets} loss.alignment=centroid
!python scripts/summarize.py --root $OUT/priority1 --split target_test --bootstrap 2000

## 6. Priority experiment 2: 2³ factorial + controls; R2.6 matched alignment  ·  ~550 GPU-h

In [ ]:
P1 = f'data.root={NIH} data.split_dir={SPLITS}/nih_shared_seed42 target.root={CHEXPERT} target.split_dir={SPLITS}/chexpert_shared_seed42'
!python scripts/run_grid.py --grid configs/grid_factorial_transfer.yaml --out $OUT --set {P1}
!python scripts/summarize.py --root $OUT/grid_factorial_transfer --split target_test --bootstrap 2000
!python scripts/factorial_analysis.py --grid configs/grid_factorial_transfer.yaml --per-run $OUT/grid_factorial_transfer/tables_target_test/per_run.csv \
     --outcome macro_auroc latent_test.effective_rank latent_test.mean_abs_offdiag_corr pca_drift.retained_variance_fixed_basis
!python scripts/run_grid.py --grid configs/grid_alignment_matched.yaml --out $OUT --set {P1}
!python scripts/summarize.py --root $OUT/grid_alignment_matched --split target_test --reference source_only --bootstrap 2000

## 7. Localisation with RSNA expert boxes (R4.9)

In [ ]:
# NIH model explained on RSNA Lung Opacity boxes; RSNA images whose NIH patient was in NIH training are excluded first
import pandas as pd
ann = pd.read_csv(f'{OUT}/annotations/rsna_boxes.csv')
ex = pd.read_csv(f'{OUT}/exclusions/rsna_vs_nih.csv')
ann = ann[~ann.path.isin(set(ex.loc[ex.in_nih_train, 'path']))]
ann.to_csv(f'{OUT}/annotations/rsna_boxes_heldout.csv', index=False)
run = f'{OUT}/grid_baselines_nih/robotxdpp/seed42'
for m in ['gradcam', 'gradcam++', 'scorecam']:
    !python scripts/explain_localization.py --run $run --config configs/nih_robotxdpp.yaml --annotations $OUT/annotations/rsna_boxes_heldout.csv --method {m} --cam-class Pneumonia

## 8. All figures from saved outputs

In [ ]:
!python scripts/make_figures.py all --root $OUT --out $OUT/figures
!find $OUT/figures -name '*.png' | head -50